In [ ]:
import sys, os, json, pickle
import pandas as pd
from pathlib import Path
import torch
from sklearn.metrics import r2_score

sys.path.append('../../src/fluprofiler')

from experiment_tools import (
    find_repo_root, default_exp_id, make_run_dirs, generate_matrix,
    load_data_and_dataloaders, evaluate_step
)
from models.architectures import fluProfiler_v0_1, fluProfiler_Config

## inference

In [69]:
season = '2025SH'
model_path = "/home/chenyh/workspace/fluProfiler/runs/reverse_tests/2025SH/v0_1/20260206_224240__v0_1__pid1314064/checkpoints/2026-02-06_23-55-23.pth"


# ---------- 1) 路径/数据 ----------
_CWD = Path.cwd().resolve()
_REPO_ROOT = find_repo_root(_CWD)     # notebook 下用 cwd 找 repo root
root_path = str(_REPO_ROOT) + "/"

data_path = root_path + "data/reverse_test/"
season_path = f"processed/test_{season}/"

exp_id = os.environ.get("FLUPROFILER_EXP_ID") or default_exp_id(_CWD, _REPO_ROOT)
tag = os.environ.get("FLUPROFILER_TAG") or "v0_1"
run_paths = make_run_dirs(_REPO_ROOT, exp_id=exp_id, tag=tag)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data_loaders = load_data_and_dataloaders(
    data_path=data_path,
    season_path=season_path,
    batch_size=8,
    sample_limit=None,
    use_artificial=False,
    test_only=True
)
test_dataloader = data_loaders["test_dataloader"]
emb_dict = data_loaders["emb_dict"]

Loading tensor: 100%|██████████| 763/763 [00:27<00:00, 27.94file/s]


In [70]:
model = torch.load(model_path, map_location=device, weights_only=False)

model.eval()
with torch.no_grad():
    test_metrics = evaluate_step(model, test_dataloader, emb_dict, device, generate_matrix, return_predictions=True)

MAE: 1.14616
MSE: 2.90935
pearson correlation: 0.59931
spearman correlation: 0.62497
R2_score: 0.22203


## Analysis

In [71]:
test_data = pd.read_csv(f'/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/test_{season}/test.csv')
test_data['prediction'] = test_metrics[5]
test_data['season'] = season

test_data.to_csv(f'./{season}_fluProfiler.csv', index=False)

In [72]:
r2_score(test_data['label'], test_data['prediction'])

0.22203039160510274

In [73]:
print('H1N1 samples: ', len(test_data[test_data['Type'] == 'H1N1']))
print('H3N2 samples: ', len(test_data[test_data['Type'] == 'H3N2']))

H1N1 samples:  2067
H3N2 samples:  2525
